# G4 · Clasificación de fuente

**Spec:** [`docs/spec_G4_codex_source_classification.md`](../docs/spec_G4_codex_source_classification.md)  |  **Bloque:** G · Caracterización  |  **Run de este set:** `ROXs42Bb_realigned`

Clasifica la fuente con una matriz hipótesis×test (umbrales congelados anti-sesgo).

| | |
|---|---|
| **Entrada** | G2/G3 + astrometría + densidad de fondo |
| **Salida (QC/productos)** | `stages/stage_g4_classification.json` |
| **Consume aguas abajo** | G5 |


## Qué hace G4 y por qué queda ambigua

G4 clasifica la fuente con una **matriz transparente hipótesis × test**: para cada hipótesis (compañero subestelar, enana marrón, estrella M asociada, en formación, fondo, contaminante, artefacto) combina los tests disponibles en una **log-verosimilitud relativa**, con **umbrales congelados** (hash anti-sesgo — el veredicto no se puede ajustar a mano).

**Tests disponibles:** T1 (astrometría en la posición ligada), T2 (fuente puntual), T7 (no es artefacto), T8, T9. **No disponibles:** T3, T4, T5, T6 (tipado espectral — pendiente de G3 — y 2ª época astrométrica).

**Resultado: clase = `substellar_companion`, robustez = `ambiguous`.**
- **CONFIRMADO — compañero REAL ligado:** artefacto excluido (log_l_rel −1004), fondo desfavorecido (−3, P_bg=6.3×10⁻⁴), contaminante (−4), vía T1+T2+T7.
- **AMBIGUO — el subtipo empata:** `substellar_companion`, `brown_dwarf` y `m_star_associated` **todas en log_l_rel 0** — sin tipado espectral (T3/T4/T6, pendiente de G3) planeta vs BD vs M no se distingue.

**Resolver la ambigüedad** requiere: G3 real (SpT/Teff/masa) + una **2ª época astrométrica** + una densidad de fondo final. Provisional.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
python -c "from musepipe.stages.stage_g4_classify import run_stage_g4; run_stage_g4('$RUN')"
```

Ligero.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_g4_classification.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -c "from musepipe.stages.stage_g4_classify import run_stage_g4; run_stage_g4(\'$RUN\')"'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_g4_classification.json', RUN_ID)
nb.show(qc, keys=['final_class', 'background_probability', 'tests_available', 'tests_unavailable'], title='G4')


## Los términos de este QC, en físico

| Término | Qué es | Por qué importa |
|---|---|---|
| `hypotheses` / `combined_ranking` | Las explicaciones posibles de lo que hay en esa posición (planeta, enana marrón, estrella M ligada, estrella de fondo…) ordenadas por cuánto las apoya la evidencia. | La clasificación es una **comparación entre hipótesis**, no una medida directa. |
| `background_probability` | Probabilidad de que una estrella no relacionada caiga por azar tan cerca en el cielo. | Es lo que descarta la coincidencia fortuita: con densidad estelar baja y separación pequeña, sale despreciable. |
| `tests_available` / `tests_unavailable` | Qué discriminantes se pudieron aplicar y cuáles no (por falta de dato, no por resultado). | Un test ausente no es evidencia en contra; que la lista sea explícita evita leerlo así. |
| `leave_one_out_stable` (LOO) | Si el ranking sobrevive al quitar **un test cada vez**. | Si al retirar un solo discriminante cambia el ganador, la clasificación se apoya en una sola pata: por eso el veredicto puede quedar «ambiguo» aunque haya un favorito. |
| `correlated_groups` | Tests que no son independientes entre sí. | Contarlos por separado inflaría artificialmente la evidencia de una hipótesis. |
| `frozen_thresholds` + `..._hash` | Los umbrales de decisión, congelados y con hash. | Impide ajustar el criterio después de ver el resultado. |


## Resultados que llevaron a la conclusión

Clase final, ranking de hipótesis y tests del `stage_g4_classification.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('G4', 'stages/stage_g4_classification.json'):
        q = nb.load_qc('stages/stage_g4_classification.json', RUN_ID)
        fc = q['final_class']
        print(f"clase = {fc['label']} | robustez = {fc['robustness']} | soportes independientes = {fc['n_independent_supports']}")
        print(f"P(fondo) = {q['background_probability']['raw']:.2e} ({q['background_probability']['source'][:40]}...)")
        print(f"tests disponibles: {q['tests_available']} | no disponibles: {q['tests_unavailable']}")
        print(f"umbrales congelados (hash anti-sesgo): {q['frozen_thresholds_hash'][:12]}")
        # Semántica oficial (musepipe.classify): la EXCLUSIÓN es un veredicto 'excludes'
        # (peso -1000), no un umbral en log_l_rel; 'disfavors' pesa -1 por test.
        try:
            from musepipe.classify import DEFAULT_WEIGHTS
            W_EXCL = DEFAULT_WEIGHTS['excludes']
        except Exception:
            W_EXCL = -1000.0   # espejo de DEFAULT_WEIGHTS (kernel sin musepipe)
        print('\nranking de hipótesis (log-verosimilitud relativa):')
        for h in q['combined_ranking']:
            v = h['log_l_rel']
            if v >= -1e-9:
                tag = '  <- EMPATE (líder)'
            elif v <= W_EXCL / 2:   # un solo 'excludes' (-1000) domina cualquier suma de ±1
                tag = '  (EXCLUIDA: >=1 test excludes)'
            else:
                tag = f'  ({abs(v):.0f} test(s) en contra, no excluida)'
            print(f"   {h['hypothesis']:22s} {v:8.0f}{tag}")


## Plot 1 — el ranking de hipótesis

Log-verosimilitud relativa de las 7 hipótesis (clip a −6; artefacto real −1004). **Tres empatan en 0** (★, verde) → compañero subestelar / BD / M asociada indistinguibles; **artefacto EXCLUIDO** (rojo: único veredicto `excludes`, peso −1000); fondo y contaminante solo **desfavorecidos** (gris: −1 por test en contra, sin exclusión). Es compañero real, pero el subtipo queda ambiguo.


In [ ]:
try:
    import matplotlib.pyplot as plt
    from musepipe.classify import DEFAULT_WEIGHTS
    W_EXCL = DEFAULT_WEIGHTS['excludes']   # -1000: semántica oficial de exclusión
    q = nb.load_qc('stages/stage_g4_classification.json', RUN_ID)
    r = q['combined_ranking']
    names = [h['hypothesis'] for h in r]; ll = [h['log_l_rel'] for h in r]
    llc = [max(v, -6) for v in ll]
    cols = ['tab:green' if v >= -1e-9 else ('tab:red' if v <= W_EXCL / 2 else '0.6') for v in ll]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    y = range(len(names))
    ax.barh(list(y), llc, color=cols)
    for i, v in enumerate(ll):
        if v >= -1e-9:
            ax.scatter(-0.15, i, marker='*', s=130, color='tab:green', zorder=5)
            ax.text(-0.35, i, 'EMPATE', va='center', ha='right', fontsize=8, color='tab:green')
        else:
            tag = f"{v:.0f}" + ('  (EXCLUIDA)' if v <= W_EXCL / 2 else '  (en contra)')
            ax.text(llc[i] - 0.1 if v > -6 else -5.9, i, tag, va='center',
                    ha='right' if v > -5 else 'left', fontsize=8)
    ax.set_yticks(list(y)); ax.set_yticklabels(names, fontsize=9); ax.invert_yaxis()
    ax.set_xlabel('log-verosimilitud relativa (clip a −6; artefacto real=−1004)'); ax.axvline(0, color='k', lw=0.6)
    ax.set_title(f"G4 · {q['final_class']['label']} / {q['final_class']['robustness']}: 3 hipótesis empatan (sin tipado espectral)")
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'g4_classify'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'ranking.png', dpi=110); print('figura ->', outdir / 'ranking.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — los tests: lo que falta para romper el empate

T1–T9: en verde los disponibles (astrometría, fuente puntual, no-artefacto…) que confirman el compañero real; en gris los **no disponibles** (T3/T4/T5/T6 = tipado espectral de G3 + 2ª época) — justo los que distinguirían planeta/BD/M.


In [ ]:
try:
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage_g4_classification.json', RUN_ID)
    avail = set(q['tests_available']); unavail = set(q['tests_unavailable'])
    tests = sorted(avail | unavail, key=lambda t: int(t[1:]))
    cols = ['tab:green' if t in avail else '0.7' for t in tests]
    fig, ax = plt.subplots(figsize=(9, 2.6))
    ax.bar(range(len(tests)), [1] * len(tests), color=cols)
    for i, t in enumerate(tests):
        ax.text(i, 0.5, t + ('\ndisp.' if t in avail else '\nfalta'), ha='center', va='center',
                fontsize=8, color='w' if t in avail else 'k', weight='bold')
    ax.set_xticks([]); ax.set_yticks([]); ax.set_ylim(0, 1)
    ax.set_title('G4 · tests disponibles (verde) vs faltantes (gris: tipado espectral + 2ª época)')
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'g4_classify'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'tests.png', dpi=110); print('figura ->', outdir / 'tests.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **Clase = `substellar_companion`, robustez = `ambiguous`**: es compañero REAL ligado (artefacto −1004, fondo P=6.3e-4, contaminante excluidos), pero subestelar/BD/M **empatan** sin tipado espectral.
- Resolver la ambigüedad requiere G3 real (SpT/Teff/masa) + 2ª época astrométrica + densidad de fondo final.
- Umbrales **congelados** (hash anti-sesgo): el veredicto no se puede ajustar a mano.


## Conclusión (registrada)

**G4: `substellar_companion` / `ambiguous` — compañero real ligado, subtipo sin resolver.**

- **Fecha:** 2026-07-08 (provisional).
- **Confirmado:** compañero real (artefacto −1004, fondo −3, contaminante −4 excluidos) vía T1+T2+T7.
- **Ambiguo:** subestelar / BD / M asociada empatan en log_l_rel 0 (sin tipado espectral).
- **Falta:** T3/T4/T6 (tipado de G3, diferido) + 2ª época astrométrica + densidad de fondo final.
- **Anti-sesgo:** umbrales congelados (hash).
- **Downstream:** G5 sintetiza (clase provisional ambigua).
